## Example 1: asgiref functionality test 1

In [ ]:
import asgiref
from asgiref.sync import sync_to_async, async_to_sync
import asyncio

# Example 1: Basic asgiref sync/async conversion
print("Example 1: Testing basic asgiref sync/async conversion")

try:
    print("asgiref module imported successfully")
    print("ASGI reference implementation with sync/async utilities")
    
    # Check available components
    attrs = [attr for attr in dir(asgiref) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Test sync_to_async decorator
    def sync_function(name, delay=0.01):
        import time
        time.sleep(delay)
        return f"Hello {name} from sync function"
    
    # Convert sync function to async
    async_function = sync_to_async(sync_function)
    
    print("Sync function converted to async")
    
    # Test async_to_sync decorator
    async def async_function_orig(name):
        await asyncio.sleep(0.01)
        return f"Hello {name} from async function"
    
    # Convert async function to sync
    sync_function_converted = async_to_sync(async_function_orig)
    
    print("Async function converted to sync")
    
    # Test both conversions
    print("Testing conversions:")
    
    # Test sync_to_async
    async def test_sync_to_async():
        result = await async_function("World", delay=0.01)
        return result
    
    async_result = await test_sync_to_async()
    print(f"sync_to_async result: {async_result}")
    
    try:
        sync_function_converted("Python")
        raise AssertionError(
            "async_to_sync should have refused to run inside the notebook's event loop"
        )
    except RuntimeError as expected_error:
        assert "event loop" in str(expected_error), expected_error
        print(f"async_to_sync correctly refused inside a running loop: {expected_error}")
    
    print("ASGI sync/async conversion working")
    print("asgiref example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgiref example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: asgiref functionality test 2

In [ ]:
from asgiref.local import Local
from asgiref.sync import sync_to_async
import asyncio
import threading

# Example 2: ASGI local context management
print("Example 2: Testing ASGI local context management")

try:
    # Create local context storage
    local = Local()
    
    print("ASGI Local context storage created")
    
    # Test setting and getting values in local context
    def set_local_value(key, value):
        setattr(local, key, value)
        return f"Set {key} = {value}"
    
    def get_local_value(key):
        return getattr(local, key, None)
    
    # Convert to async for testing
    async_set = sync_to_async(set_local_value)
    async_get = sync_to_async(get_local_value)
    
    # Test local context in async environment
    async def test_local_context():
        # Set value in async context
        await async_set('user_id', 12345)
        await async_set('request_id', 'req-abc-123')
        
        # Get values from same context
        user_id = await async_get('user_id')
        request_id = await async_get('request_id')
        
        print(f"Retrieved user_id: {user_id}")
        print(f"Retrieved request_id: {request_id}")
        
        return user_id, request_id
    
    # Test the local context
    result = await test_local_context()
    print(f"Local context test result: {result}")
    
    # Test context isolation between different async tasks
    async def isolated_task(task_id):
        await async_set('task_id', task_id)
        await asyncio.sleep(0.01)  # Simulate work
        retrieved_id = await async_get('task_id')
        return f"Task {task_id}: retrieved {retrieved_id}"
    
    # Run multiple tasks to test isolation
    tasks = [isolated_task(i) for i in range(3)]
    isolation_results = await asyncio.gather(*tasks)
    
    print("Context isolation test:")
    for result in isolation_results:
        print(f"  {result}")
    
    print("ASGI local context management working")
    print("asgiref example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgiref example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: asgiref functionality test 3

In [ ]:
from asgiref.compatibility import guarantee_single_callable
import asgiref
import asyncio

# Example 3: ASGI compatibility utilities
print("Example 3: Testing ASGI compatibility utilities")

try:
    print("Testing ASGI compatibility and callable utilities")
    
    # Test guarantee_single_callable - ensures ASGI app is single callable
    print("Testing ASGI callable compatibility:")
    
    # ASGI 2 style application (double callable)
    def asgi2_app(scope):
        async def asgi_coroutine(receive, send):
            if scope["type"] == "http":
                await send({
                    "type": "http.response.start",
                    "status": 200,
                    "headers": [[b"content-type", b"text/plain"]]
                })
                await send({
                    "type": "http.response.body",
                    "body": b"ASGI 2 style app"
                })
        return asgi_coroutine
    
    # ASGI 3 style application (single callable)
    async def asgi3_app(scope, receive, send):
        if scope["type"] == "http":
            await send({
                "type": "http.response.start",
                "status": 200,
                "headers": [[b"content-type", b"text/plain"]]
            })
            await send({
                "type": "http.response.body",
                "body": b"ASGI 3 style app"
            })
    
    print("ASGI 2 and ASGI 3 style applications created")
    
    # Test compatibility conversion
    try:
        converted_app = guarantee_single_callable(asgi2_app)
        print("ASGI 2 app converted to single callable")
        print(f"Converted app type: {type(converted_app).__name__}")
        
        # Test that ASGI 3 app passes through unchanged
        unchanged_app = guarantee_single_callable(asgi3_app)
        print(f"ASGI 3 app unchanged: {unchanged_app is asgi3_app}")
        
    except Exception as e:
        print(f"Compatibility conversion: {type(e).__name__}")
        raise
    
    # Test version detection and compatibility
    print("ASGI version compatibility:")
    print("• ASGI 2: Double callable (scope)(receive, send)")
    print("• ASGI 3: Single callable (scope, receive, send)")
    print("• asgiref handles both formats seamlessly")
    
    # Show asgiref version compatibility
    if hasattr(asgiref, '__version__'):
        print(f"asgiref version: {asgiref.__version__}")
    
    # Test scope validation concepts
    sample_scope = {
        "type": "http",
        "method": "GET",
        "path": "/test",
        "query_string": b"",
        "headers": []
    }
    
    print(f"Sample ASGI scope: {list(sample_scope.keys())}")
    
    print("ASGI compatibility utilities working")
    print("asgiref example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgiref example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: asgiref functionality test 4

In [ ]:
from asgiref.sync import sync_to_async, async_to_sync
from asgiref.local import Local
import asyncio
import threading

# Example 4: Thread-safe async/sync operations
print("Example 4: Testing thread-safe async/sync operations")

try:
    print("Testing thread-safe operations with asgiref")
    
    # Create shared local storage
    local_storage = Local()
    
    # Test thread safety with multiple operations
    def sync_database_operation(user_id, operation):
        """Simulate a synchronous database operation"""
        import time
        time.sleep(0.01)  # Simulate DB delay
        
        # Store in local context
        local_storage.current_user = user_id
        local_storage.operation = operation
        
        return f"DB operation '{operation}' for user {user_id}"
    
    # Convert to async
    async_db_operation = sync_to_async(sync_database_operation, thread_sensitive=True)
    
    # Test concurrent async operations
    async def process_user_request(user_id, request_type):
        # Perform async database operation
        result = await async_db_operation(user_id, request_type)
        
        # Check local context
        stored_user = getattr(local_storage, 'current_user', None)
        stored_operation = getattr(local_storage, 'operation', None)
        
        return {
            'user_id': user_id,
            'request_type': request_type,
            'db_result': result,
            'local_user': stored_user,
            'local_operation': stored_operation
        }
    
    # Test multiple concurrent requests
    print("Testing concurrent request processing:")
    
    requests = [
        (101, 'login'),
        (102, 'profile_update'), 
        (103, 'data_fetch'),
        (104, 'logout')
    ]
    
    # Process all requests concurrently
    tasks = [process_user_request(user_id, req_type) for user_id, req_type in requests]
    results = await asyncio.gather(*tasks)
    
    # Display results
    for result in results:
        print(f"User {result['user_id']}: {result['request_type']} -> {result['db_result'][:30]}...")
        print(f"  Local context: user={result['local_user']}, op={result['local_operation']}")
    
    # Test thread pool configuration
    print("\nThread pool configuration:")
    
    # Test with different thread sensitivity settings
    thread_insensitive = sync_to_async(sync_database_operation, thread_sensitive=False)
    thread_sensitive = sync_to_async(sync_database_operation, thread_sensitive=True)
    
    print("Thread-sensitive and thread-insensitive conversions created")
    
    # Test execution
    result1 = await thread_insensitive(999, 'test_insensitive')
    result2 = await thread_sensitive(998, 'test_sensitive')
    
    print(f"Thread insensitive result: {result1}")
    print(f"Thread sensitive result: {result2}")
    
    print("Thread-safe async/sync operations working")
    print("asgiref example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgiref example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: asgiref functionality test 5

In [ ]:
import asgiref
from asgiref.sync import sync_to_async, async_to_sync
from asgiref.local import Local
import asyncio

# Example 5: asgiref package capabilities and Django integration
print("Example 5: asgiref package capabilities and integration")

try:
    print("asgiref provides ASGI reference implementation and utilities")
    
    # Package analysis
    attrs = [attr for attr in dir(asgiref) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Show version information
    if hasattr(asgiref, '__version__'):
        print(f"asgiref version: {asgiref.__version__}")
    
    # Demonstrate real-world integration patterns
    print("\nReal-world integration examples:")
    
    # Django-style database integration
    class MockDjangoORM:
        """Mock Django ORM-style database operations"""
        
        def __init__(self):
            self.users = {
                1: {'name': 'Alice', 'email': 'alice@example.com'},
                2: {'name': 'Bob', 'email': 'bob@example.com'},
                3: {'name': 'Charlie', 'email': 'charlie@example.com'}
            }
        
        def get_user(self, user_id):
            """Synchronous user lookup"""
            import time
            time.sleep(0.01)  # Simulate database query
            return self.users.get(user_id)
        
        def create_user(self, name, email):
            """Synchronous user creation"""
            import time
            time.sleep(0.02)  # Simulate database write
            user_id = max(self.users.keys()) + 1
            self.users[user_id] = {'name': name, 'email': email}
            return user_id
    
    # Create mock ORM instance
    orm = MockDjangoORM()
    
    # Convert ORM methods to async
    async_get_user = sync_to_async(orm.get_user)
    async_create_user = sync_to_async(orm.create_user)
    
    print("Django-style ORM methods converted to async")
    
    # Test async ORM operations
    async def test_async_orm():
        # Get existing user
        user = await async_get_user(1)
        print(f"Retrieved user: {user}")
        
        # Create new user
        new_user_id = await async_create_user("Diana", "diana@example.com")
        new_user = await async_get_user(new_user_id)
        print(f"Created user {new_user_id}: {new_user}")
        
        return new_user_id
    
    new_id = await test_async_orm()
    
    # Show key capabilities
    print(f"\nKey asgiref capabilities:")
    capabilities = [
        "🔄 Sync to async function conversion (sync_to_async)",
        "🔃 Async to sync function conversion (async_to_sync)",
        "🧵 Thread-safe execution in async contexts",
        "📦 Local context storage for request isolation",
        "🔧 ASGI 2/3 compatibility layer",
        "🌐 Django async views and middleware support",
        "⚡ Executor management for blocking operations",
        "🛡️ Context variable propagation across threads"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Integration patterns
    print(f"\nCommon integration patterns:")
    patterns = [
        "Django async views calling sync ORM methods",
        "FastAPI endpoints using synchronous libraries", 
        "ASGI middleware bridging sync/async code",
        "Background task processing with mixed code",
        "Legacy codebase gradual async migration",
        "Database connection pooling in async apps",
        "File I/O operations in async web servers",
        "Third-party API integration in async frameworks"
    ]
    
    for pattern in patterns:
        print(f"• {pattern}")
    
    # Performance considerations
    print(f"\nPerformance considerations:")
    considerations = [
        "thread_sensitive=True for database connections",
        "thread_sensitive=False for CPU-bound operations",
        "Local() for request-scoped variable storage",
        "Proper executor configuration for blocking I/O",
        "Context manager cleanup for resources"
    ]
    
    for consideration in considerations:
        print(f"• {consideration}")
    
    print(f"\nasgiref enables seamless sync/async integration")
    print("asgiref example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgiref example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")